# Homework 20: Tokenization and Next-Token Prediction

**Audience.** Students who know Python, NumPy, PyTorch linear layers, and cross-entropy, but have not studied transformers.

**Prerequisites.** Strings and dictionaries; tensor shapes; matrix multiplication; `nn.Embedding`; categorical cross-entropy.

**Learning goals.** By the end of this notebook, you will be able to:

- split text into word and punctuation tokens;
- construct a reproducible vocabulary and translate between tokens and integer IDs;
- explain token and positional embeddings;
- construct inputs and one-position-shifted targets for next-token prediction.

All implementations are complete. Read each code cell before running it and predict important shapes and values.


## Outline

1. Why a model needs tokens
2. A transparent word-and-punctuation tokenizer
3. Integer IDs and embeddings
4. Context windows and shifted targets
5. Notebook checkpoints


In [1]:
# S1: Imports and reproducibility
import math
import re
from collections import Counter

import torch
from torch import nn

SEED = 158
_ = torch.manual_seed(SEED)


## 1. Text must become a sequence

A neural network cannot consume a Python string directly. We first choose the units in the sequence. Character tokens make a very small vocabulary, but force the model to learn spelling. Word tokens carry more meaning per position, but require a larger vocabulary.

For this course model, punctuation will be separate from words. This keeps periods and quotation marks visible for our later interpretability experiments.


In [2]:
# S2: Compare character tokens with word-and-punctuation tokens
sentence = 'Ava said, "The blue kite can fly!"'
character_tokens = list(sentence)

TOKEN_PATTERN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]")

def tokenize(text):
    return TOKEN_PATTERN.findall(text)

word_tokens = tokenize(sentence)

print("character tokens:", character_tokens)
print("word/punctuation tokens:", word_tokens)
print("lengths:", len(character_tokens), "and", len(word_tokens))


character tokens: ['A', 'v', 'a', ' ', 's', 'a', 'i', 'd', ',', ' ', '"', 'T', 'h', 'e', ' ', 'b', 'l', 'u', 'e', ' ', 'k', 'i', 't', 'e', ' ', 'c', 'a', 'n', ' ', 'f', 'l', 'y', '!', '"']
word/punctuation tokens: ['Ava', 'said', ',', '"', 'The', 'blue', 'kite', 'can', 'fly', '!', '"']
lengths: 34 and 11


The regular expression has three alternatives: a word (possibly containing an apostrophe), an integer, or one non-whitespace punctuation character. It is intentionally simpler than a production tokenizer such as BPE.


In [3]:
# S3: Construct a deterministic vocabulary from a tiny corpus
stories = [
    "Ava found a blue kite. The kite flew high!",
    "Ben found a red ball. The ball rolled away.",
    'Cora said, "Please bring the blue ball."',
    "A dog found the ball. Ava thanked the dog.",
]

special_tokens = ["<pad>", "<unk>", "<bos>", "<eos>"]
counts = Counter(token for story in stories for token in tokenize(story))

# Frequency first; alphabetical order breaks ties reproducibly.
ordinary_tokens = sorted(counts, key=lambda token: (-counts[token], token))
vocabulary = special_tokens + ordinary_tokens
stoi = {token: index for index, token in enumerate(vocabulary)}
itos = {index: token for token, index in stoi.items()}

print("vocabulary size:", len(vocabulary))
print("first 16 entries:", vocabulary[:16])
print("IDs for '.', 'ball', and '<bos>':", stoi["."], stoi["ball"], stoi["<bos>"])


vocabulary size: 29
first 16 entries: ['<pad>', '<unk>', '<bos>', '<eos>', '.', 'ball', 'found', 'the', '"', 'Ava', 'The', 'a', 'blue', 'dog', 'kite', '!']
IDs for '.', 'ball', and '<bos>': 4 5 2


`<bos>` marks the beginning of a story and `<eos>` marks its end. `<unk>` represents a word outside this vocabulary. `<pad>` is useful when examples of different lengths must share a batch, although we will mostly use fixed-length windows.


In [4]:
# S4: Encode text as IDs and decode IDs back to tokens
def encode(text, add_special_tokens=True):
    ids = [stoi.get(token, stoi["<unk>"]) for token in tokenize(text)]
    if add_special_tokens:
        ids = [stoi["<bos>"]] + ids + [stoi["<eos>"]]
    return ids

def decode_to_tokens(ids):
    return [itos[int(index)] for index in ids]

encoded_story = encode(stories[0])
decoded_tokens = decode_to_tokens(encoded_story)

print("IDs:", encoded_story)
print("tokens:", decoded_tokens)
print("unknown-word example:", decode_to_tokens(encode("Ava found a spaceship.")))


IDs: [2, 9, 6, 11, 12, 14, 4, 10, 14, 23, 24, 15, 3]
tokens: ['<bos>', 'Ava', 'found', 'a', 'blue', 'kite', '.', 'The', 'kite', 'flew', 'high', '!', '<eos>']
unknown-word example: ['<bos>', 'Ava', 'found', 'a', '<unk>', '.', '<eos>']


## 2. Embeddings turn IDs into vectors

An ID is only a label: token 12 is not numerically “larger” than token 5. `nn.Embedding` stores one learned vector for each token. A second embedding identifies each position in the context window.


In [5]:
# S5: Token embeddings and positional embeddings
embedding_dimension = 8
maximum_context = 12

token_embedding = nn.Embedding(len(vocabulary), embedding_dimension)
position_embedding = nn.Embedding(maximum_context, embedding_dimension)

example_ids = torch.tensor([encoded_story[:maximum_context]])
positions = torch.arange(example_ids.shape[1])

token_vectors = token_embedding(example_ids)
position_vectors = position_embedding(positions)
input_vectors = token_vectors + position_vectors

print("ID shape:", tuple(example_ids.shape))
print("token-vector shape:", tuple(token_vectors.shape))
print("position-vector shape:", tuple(position_vectors.shape))
print("combined-input shape:", tuple(input_vectors.shape))


ID shape: (1, 12)
token-vector shape: (1, 12, 8)
position-vector shape: (12, 8)
combined-input shape: (1, 12, 8)


The position embedding has no batch dimension, but PyTorch broadcasts it across the batch when it is added to the token embeddings. The same word at two positions gets the same token vector but a different combined input vector.


In [6]:
# S6: Verify the roles of the two embedding tables
repeated_ids = torch.tensor([[stoi["ball"], stoi["ball"]]])
repeated_token_vectors = token_embedding(repeated_ids)
repeated_inputs = repeated_token_vectors + position_embedding(torch.arange(2))

same_token_vector = torch.allclose(
    repeated_token_vectors[0, 0], repeated_token_vectors[0, 1]
)
same_combined_vector = torch.allclose(repeated_inputs[0, 0], repeated_inputs[0, 1])

print("same token vector:", same_token_vector)
print("same vector after positions are added:", same_combined_vector)


same token vector: True
same vector after positions are added: False


## 3. Next-token examples

If the input is `[<bos>, Ava, found, a]`, the desired output is `[Ava, found, a, blue]`. Each target is the token immediately to the right of the corresponding input token.


In [7]:
# S7: Make fixed-length input/target windows
token_stream = []
for story in stories:
    token_stream.extend(encode(story))
token_stream = torch.tensor(token_stream)

context_length = 6

def make_window(stream, start, length=context_length):
    inputs = stream[start : start + length]
    targets = stream[start + 1 : start + length + 1]
    return inputs, targets

inputs, targets = make_window(token_stream, start=0)

print("input IDs:   ", inputs.tolist())
print("target IDs:  ", targets.tolist())
print("input tokens:", decode_to_tokens(inputs))
print("targets:     ", decode_to_tokens(targets))
assert torch.equal(inputs[1:], targets[:-1])


input IDs:    [2, 9, 6, 11, 12, 14]
target IDs:   [9, 6, 11, 12, 14, 4]
input tokens: ['<bos>', 'Ava', 'found', 'a', 'blue', 'kite']
targets:      ['Ava', 'found', 'a', 'blue', 'kite', '.']


A transformer will produce one vocabulary-sized logit vector at every position. Cross-entropy compares each vector with the corresponding target ID. Before training, equal logits assign probability `1 / vocabulary_size` to every token, so the loss is `log(vocabulary_size)`.


In [8]:
# S8: Shape of a next-token loss computation
uniform_logits = torch.zeros(1, context_length, len(vocabulary))
uniform_loss = nn.functional.cross_entropy(
    uniform_logits.reshape(-1, len(vocabulary)),
    targets.reshape(-1),
)

print("logit shape:", tuple(uniform_logits.shape))
print("uniform loss:", round(float(uniform_loss), 6))
print("log(vocabulary size):", round(math.log(len(vocabulary)), 6))


logit shape: (1, 6, 29)
uniform loss: 3.367296
log(vocabulary size): 3.367296


## Notebook checkpoints

These named values are designed for exact code-comprehension questions. Before running the next cell, trace S2–S8 and predict each entry.


In [9]:
# S9: Deterministic checkpoint record
checkpoint_20 = {
    "word_tokens": word_tokens,
    "vocabulary_size": len(vocabulary),
    "period_id": stoi["."],
    "input_tokens": decode_to_tokens(inputs),
    "target_tokens": decode_to_tokens(targets),
    "combined_input_shape": tuple(input_vectors.shape),
    "same_token_vector": same_token_vector,
    "same_combined_vector": same_combined_vector,
}
checkpoint_20


{'word_tokens': ['Ava',
  'said',
  ',',
  '"',
  'The',
  'blue',
  'kite',
  'can',
  'fly',
  '!',
  '"'],
 'vocabulary_size': 29,
 'period_id': 4,
 'input_tokens': ['<bos>', 'Ava', 'found', 'a', 'blue', 'kite'],
 'target_tokens': ['Ava', 'found', 'a', 'blue', 'kite', '.'],
 'combined_input_shape': (1, 12, 8),
 'same_token_vector': True,
 'same_combined_vector': False}

## Pitfall and extension

**Pitfall.** Splitting text with `text.split()` would leave `kite.` as one token and `kite!` as another. The regular expression prevents that accidental vocabulary growth.

**Optional extension.** Compare this vocabulary with a character vocabulary. Count how many sequence positions each representation needs for the four stories. Production language models often use subword tokenization as a compromise between the two.
